## 🎨🖼️🏖️ Plot the shoreline 

This notebook makes grids of shoreline plots, taking different slices through the 3D cosmic shoreline space.

In [ ]:
format = 'paper'
if format == 'poster':
    annotation_font_size = 4
    figsize=(10, 3)
    mapsize=None
if format == 'paper':
    annotation_font_size = 4
    figsize=(10,3)
    mapsize=None

In [ ]:
from shoreline import * 

## Load populations + posteriors.

The reads in a set of populations with atmosphere labels attached, as well as a posterior containing samples from the shoreline parameters.

In [ ]:
# load organized + labeled populations 
collection_of_pops = {}
subset = 'all'
fluxlimits = ['any', 'no-magma', 'no-magma-no-freeze']
for fluxlimit in fluxlimits:
    collection_of_pops[fluxlimit] = load_organized_populations(subset=subset, fluxlimit=fluxlimit)[f'{subset}+{fluxlimit}']

In [ ]:
# load the posterior and store in shoreline object
subset='all'
uncertainties=True
collection_of_posteriors = {}
for fluxlimit in fluxlimits:
    collection_of_posteriors[fluxlimit] = az.from_netcdf(f'posteriors/{subset}+{fluxlimit}+uncertainties={uncertainties}+numpyro.nc')

## Decide Which Planets to Annotate

In [ ]:
# define some planets we might want to annotate

annotation_options = dict(
    many=[
        "Mercury",
        "Venus",
        "Earth",
        "Mars",
        "Jupiter",
        "Saturn",
        "Uranus",
        "Neptune",
        "Moon",
        "Pluto",
        "Eris",
        "Haumea",
        "Makemake",
        "Ceres",
        "Titan",
        "55 Cnc e",
        "TOI 561b",
        "LHS 3844 b",
        "GJ367b",
        "TOI-1685b",
        "GJ1252b",
        "GJ486b",
        "GJ1132b",
        "LTT1445Ab",
        "TOI-1468 b",
        "LHS 1140 c",
        "Trappist-1b",
        "Trappist-1c",
        "GJ 3929b",
        "LTT 3780b",
        "GJ-3929 b",
        "LTT-1445 A c",
        "LTT-1445 A b",
        "LHS-1140 b",
        "TOI-198 b",
        "TOI-406 c",
        "TOI-771 b",
        "HD 260655 c",
        "TOI-244 b",
        "LHS 1478 b",
        "Kepler-10b",
        "Kepler-78b",
        "K2-141 b",
        "L 98-59b",
        "GJ 1214b",
        "K2-18b",
        "TOI-700d",
        "TOI-700e",
        "Kepler-62e",
        "Kepler-62f",
        "L 98-59c",
        "L 98-59d",
    ]
    + [f"Trappist-1{k}" for k in "defgh"],
    few=[
        "Mercury",
        "Venus",
        "Earth",
        "Mars",
        "Jupiter",
        "Saturn",
        "Uranus",
        "Neptune",
        "Moon",
        "Pluto",
        "Titan",
        "Ceres",
        "TOI 561b",
        "LHS-1140 b",
    ]
    + [f"Trappist-1{k}" for k in "bcdefgh"],
)

In [ ]:
# modify how planets get annotated in plots 
from exoatlas.visualizations import * 
for k, v in clean_pops(collection_of_pops).items():
    v.annotate_planets = True 
    v.annotate_kw = dict(rotation=0, rotation_mode='anchor', format="    {}", fontsize=annotation_font_size)


## Make some plots.

Let's make a few different visualizations, some static and some animated. 

### Grid of shoreline slices. 

First, we'll make a grid showing different slices of the 3D shoreline space side-by-side. For these plots, we'll use the `no-magma` posterior, but show all planets (even some of those that weren't used in the fit). These plots will appear directly in the paper to show all three possible projects of the cosmic shoreline fit.

In [ ]:

# show all planets but use the `no-magma` fit posteriors
pops = collection_of_pops['any']
fluxlimit = 'no-magma'

# decide how many planets to label
for annotation_type, planets_to_annotate in annotation_options.items():

    # set the planets to label
    for k, v in clean_pops(collection_of_pops).items():
        v.annotate_kw['names']=planets_to_annotate

    # loop through rotations
    for order, letters in zip(['vfL', 'fLv', 'Lvf'], ['abcd', 'efgh', 'ijkl']):
        posterior = collection_of_posteriors[fluxlimit]

        # create a standard shoreline cube plot
        m = ShorelineStandardMap(order=order, posterior=posterior, kludge=True, invisible_fraction=0.8)
        # split that cube into multiple slices
        g = SliceGridGallery(m, N=4, figsize=figsize, mapsize=mapsize, dpi=300)
        # plot the planets
        g.build(pops)
        # refine the plots, including adding probabilities
        g.refine(probability=True, limits=True)
        # add a colorbar for the shoreline probability
        g.maps[len(g.maps)-1].add_colorbar()
        g.add_panel_labels(letters=letters)
        # save the figure
        plt.savefig(f'figures/shoreline+{order}+{fluxlimit}+all-planets+{annotation_type}-annotations.pdf')

    


In [ ]:
!cp figures/shoreline+*+no-magma+all-planets+few-annotations.pdf paper-figures/.

Now, mostly for talks, let's produce versions of this cosmic shoreline plot using each of the three kinds of fit (`any`, `no-magma`, `no-magma-no-freeze`). We'll keep 

In [ ]:

# loop over fits 
for fluxlimit in fluxlimits:
    pops = collection_of_pops[fluxlimit]
    posterior = collection_of_posteriors[fluxlimit]

    # decide how many planets to label
    for annotation_type, planets_to_annotate in annotation_options.items():
        # set the planets to label
        for k, v in clean_pops(pops).items():
            v.annotate_kw['names']=planets_to_annotate

        order = 'vfL'
        probabilty = True
        # create a standard shoreline cube plot
        m = ShorelineStandardMap(order=order, posterior=posterior, kludge=True, invisible_fraction=0.8)
        # split that cube into multiple slices
        g = SliceGridGallery(m, N=4, figsize=figsize, mapsize=mapsize, dpi=300)
        # plot the planets
        g.build(pops)
        # refine the plots, including adding probabilities
        g.refine(probability=True, limits=True)
        g.add_panel_labels()
        # save the figure
        plt.savefig(f'figures/shoreline+{order}+{fluxlimit}+{annotation_type}-annotations.pdf')

In [ ]:
!cp figures/shoreline+vfL+no-magma-no-freeze+few-annotations.pdf paper-figures/

### Animated shoreline slices. 

Next, let's animate these same slices, flipping through them in time, rather than seeing them side by side.

In [ ]:
pops = collection_of_pops['any']
posterior = collection_of_posteriors['no-magma']


# decide how many planets to label
for annotation_type, planets_to_annotate in annotation_options.items():

    # set the planets to label
    for k, v in clean_pops(pops).items():
        v.annotate_kw['names']=planets_to_annotate


    for order in ['vfL', 'fLv', 'Lvf']:
        m = ShorelineStandardMap(order=order, posterior=posterior)
        a = SliceAnimatedGallery(m, N=4, dpi=600, figsize=(4,4))
        a.animate(pops, filename=f'figures/shoreline-{order}+{annotation_type}-annotations-animated.mp4', refine_kw=dict(probability=True, limits=True))
        plt.savefig(f'figures/shoreline-{order}+{annotation_type}-annotations-animated-static-frame.pdf')

In [ ]:
!cp figures/shoreline-vfL+few-annotations-animated.mp4 paper-figures/.
!cp figures/shoreline-vfL+few-annotations-animated-static-frame.pdf paper-figures/.